In [1]:
import pandas as pd

nba = pd.read_csv("NBA_stats.csv")
nba.head(n=5)

,Rk,Player,Pos,Age,Tm,G,GS,MP,FG,FGA,...,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1,Álex Abrines\abrinal01,SG,23,OKC,68,6,15.5,2.0,5.0,...,0.898,0.3,1.0,1.3,0.6,0.5,0.1,0.5,1.7,6.0
1,2,Quincy Acy\acyqu01,PF,26,TOT,38,1,14.7,1.8,4.5,...,0.750,0.5,2.5,3.0,0.5,0.4,0.4,0.6,1.8,5.8
2,2,Quincy Acy\acyqu01,PF,26,DAL,6,0,8.0,0.8,2.8,...,0.667,0.3,1.0,1.3,0.0,0.0,0.0,0.3,1.5,2.2
3,2,Quincy Acy\acyqu01,PF,26,BRK,32,1,15.9,2.0,4.8,...,0.754,0.6,2.8,3.3,0.6,0.4,0.5,0.6,1.8,6.5
4,3,Steven Adams\adamsst01,C,23,OKC,80,80,29.9,4.7,8.2,...,0.611,3.5,4.2,7.7,1.1,1.1,1.0,1.8,2.4,11.3


In [2]:
wnba = pd.read_csv("WNBA_stats.csv")
wnba.head(n=5)

,Name,Team,Pos,Height,Weight,BMI,Birth_Place,Birthdate,Age,College,...,OREB,DREB,REB,AST,STL,BLK,TO,PTS,DD2,TD3
0,Aerial Powers,DAL,F,183,71.0,21.200991,US,"January 17, 1994",23,Michigan State,...,6,22,28,12,3,6,12,93,0,0
1,Alana Beard,LA,G/F,185,73.0,21.329438,US,"May 14, 1982",35,Duke,...,19,82,101,72,63,13,40,217,0,0
2,Alex Bentley,CON,G,170,69.0,23.875433,US,"October 27, 1990",26,Penn State,...,4,36,40,78,22,3,24,218,0,0
3,Alex Montgomery,SAN,G/F,185,84.0,24.543462,US,"December 11, 1988",28,Georgia Tech,...,35,134,169,65,20,10,38,188,2,0
4,Alexis Jones,MIN,G,175,78.0,25.469388,US,"August 5, 1994",23,Baylor,...,3,9,12,12,7,0,14,50,0,0


## NBA vs WNBA — style of play comparisons

The NBA dataset reports per-game averages; the WNBA dataset reports season totals, so WNBA stats are divided by `Games Played` to put both leagues on a per-game scale. Field goal and three-point percentages are rescaled to 0–100 so both leagues share a common axis.

In [3]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected"
LEAGUE_COLORS = {"NBA": "#66c2a5", "WNBA": "#fc8d62"}
BASE_FONT = dict(family="Arial", size=13)

def pos_bucket(p):
    if p in ('PG', 'SG', 'G'): return 'Guard'
    if p in ('SF', 'PF', 'F'): return 'Forward'
    if p == 'C': return 'Center'
    return 'Hybrid'

nba_p = nba[['Player', 'Pos', 'MP', 'FGA', 'FG', '3PA', '3P%', 'FG%', 'PTS',
             'AST', 'STL', 'BLK', 'TOV', 'ORB', 'DRB', 'TRB']].copy()
nba_p['Player'] = nba_p['Player'].str.split('\\').str[0]
nba_p['FG%'] = nba_p['FG%'] * 100
nba_p['3P%'] = nba_p['3P%'] * 100
nba_p['League'] = 'NBA'

wnba_p = pd.DataFrame({
    'Player': wnba['Name'],
    'Pos':    wnba['Pos'],
    'MP':  wnba['MIN']  / wnba['Games Played'],
    'FGA': wnba['FGA']  / wnba['Games Played'],
    'FG':  wnba['FGM']  / wnba['Games Played'],
    '3PA': wnba['3PA']  / wnba['Games Played'],
    '3P%': wnba['3P%'],
    'FG%': wnba['FG%'],
    'PTS': wnba['PTS']  / wnba['Games Played'],
    'AST': wnba['AST']  / wnba['Games Played'],
    'STL': wnba['STL']  / wnba['Games Played'],
    'BLK': wnba['BLK']  / wnba['Games Played'],
    'TOV': wnba['TO']   / wnba['Games Played'],
    'ORB': wnba['OREB'] / wnba['Games Played'],
    'DRB': wnba['DREB'] / wnba['Games Played'],
    'TRB': wnba['REB']  / wnba['Games Played'],
})
wnba_p['League'] = 'WNBA'

combined = pd.concat([nba_p, wnba_p], ignore_index=True)
combined['PosGroup'] = combined['Pos'].apply(pos_bucket)
combined['ORB_share'] = combined['ORB'] / combined['TRB'].replace(0, pd.NA)
combined.groupby('League')[['MP','FGA','PTS','3PA','FG%','AST','STL','BLK','TOV','ORB','DRB','TRB']].agg(['mean','std']).round(2).T

League      NBA   WNBA
MP  mean  19.54  19.04
    std    8.84   9.08
FGA mean   6.73   6.40
    std    4.37   3.93
PTS mean   8.19   7.70
    std    5.84   5.23
3PA mean   2.17   1.65
    std    1.89   1.64
FG% mean  44.13  42.90
    std   10.28  10.11
AST mean   1.75   1.69
    std    1.70   1.50
STL mean   0.61   0.67
    std    0.41   0.46
BLK mean   0.39   0.38
    std    0.48   0.46
TOV mean   1.08   1.23
    std    0.77   0.69
ORB mean   0.83   0.84
    std    0.76   0.73
DRB mean   2.67   2.34
    std    1.80   1.65
TRB mean   3.49   3.17
    std    2.43   2.26

### 1. Field goals attempted per game — NBA vs WNBA

Split violin plot of per-game shot volume showing the full distribution shape, embedded box, mean marker, and every individual player as a jittered point — the NBA's longer high-volume tail vs the WNBA's tighter cluster is the key visual takeaway.

In [4]:
fig = px.violin(
    combined, x='League', y='FGA', color='League',
    box=True, points='all',
    color_discrete_map=LEAGUE_COLORS,
    hover_data=['Player', 'Pos'],
)
fig.update_traces(meanline_visible=True, marker=dict(size=4, opacity=0.55), jitter=0.4)

means = combined.groupby('League')['FGA'].mean().round(2)
for league, m in means.items():
    fig.add_annotation(
        x=league, y=m, text=f'<b>μ = {m}</b>',
        showarrow=False, yshift=18,
        font=dict(size=13, color='#222'),
        bgcolor='rgba(255,255,255,0.92)', bordercolor='#666', borderwidth=1, borderpad=4,
    )

fig.update_layout(
    title=dict(
        text='<b>Field Goals Attempted per Game by League</b><br>'
             '<sub style="color:#666">NBA carries a longer high-volume tail; WNBA distribution is tighter</sub>',
        x=0.02, xanchor='left',
    ),
    yaxis_title='FGA per game', xaxis_title='',
    height=580, showlegend=False,
    margin=dict(t=100, l=70, r=30, b=40),
    font=BASE_FONT,
)
fig.show()

### 2. Points per game — distribution with mean lines

Overlapping density histograms with violin marginals and a dotted reference line at each league's mean, so the central tendency, the spread, and any high-scoring outliers are all readable on a single chart.

In [5]:
fig = px.histogram(
    combined, x='PTS', color='League',
    color_discrete_map=LEAGUE_COLORS,
    barmode='overlay', opacity=0.55, nbins=40,
    histnorm='probability density',
    marginal='violin',
)
means = combined.groupby('League')['PTS'].mean()
for league, m in means.items():
    fig.add_vline(
        x=m, line_dash='dot', line_color=LEAGUE_COLORS[league],
        line_width=2, opacity=0.9,
        annotation_text=f'{league} mean: {m:.1f}',
        annotation_position='top',
    )
fig.update_layout(
    title=dict(
        text='<b>Distribution of Points per Game</b><br>'
             '<sub style="color:#666">Dotted lines mark the per-league mean</sub>',
        x=0.02, xanchor='left',
    ),
    xaxis_title='Points per game', yaxis_title='Density',
    height=580,
    margin=dict(t=100, l=70, r=30, b=40),
    font=BASE_FONT,
)
fig.show()

### 3. Three-point attempts vs points scored — NBA vs WNBA

Scatter plot of 3-point attempts per game against points scored per game with marginal violins on each axis and a per-league OLS trendline, so you can see both the individual player cloud and how perimeter volume translates into scoring in each league.

In [6]:
fig = px.scatter(
    combined, x='3PA', y='PTS', color='League',
    color_discrete_map=LEAGUE_COLORS,
    trendline='ols',
    marginal_x='violin', marginal_y='violin',
    hover_data=['Player', 'Pos', 'FG%', '3P%'],
    opacity=0.7,
)
fig.update_traces(
    marker=dict(size=8, line=dict(width=0.5, color='white')),
    selector=dict(mode='markers'),
)
fig.update_layout(
    title=dict(
        text='<b>Three-Point Attempts vs Points per Game</b><br>'
             '<sub style="color:#666">Per-league trendline shows how perimeter volume converts to scoring</sub>',
        x=0.02, xanchor='left',
    ),
    xaxis_title='3PA per game',
    yaxis_title='Points per game',
    height=640,
    margin=dict(t=100, l=70, r=30, b=40),
    font=BASE_FONT,
)
fig.show()

### 4. Field goal percentage — cumulative distribution

Empirical cumulative distribution (ECDF) of FG% for each league: at any horizontal point you can read off the percentage of players who shoot at or below that accuracy, and a curve sitting further to the right indicates a higher-shooting league overall.

In [7]:
fig = px.ecdf(
    combined[combined['FG%'].between(1, 99)],
    x='FG%', color='League',
    color_discrete_map=LEAGUE_COLORS,
)
fig.update_traces(line=dict(width=3))
fig.update_layout(
    title=dict(
        text='<b>Cumulative Distribution of Field Goal Percentage</b><br>'
             '<sub style="color:#666">Curves further to the right = higher-shooting league</sub>',
        x=0.02, xanchor='left',
    ),
    xaxis_title='FG%',
    yaxis_title='Share of players at or below',
    yaxis_tickformat='.0%',
    height=520,
    margin=dict(t=100, l=70, r=30, b=40),
    font=BASE_FONT,
)
fig.show()

### 5. Position composition — NBA vs WNBA

Two side-by-side donut charts, one per league, each showing the full breakdown of players across the four position groups (Guard, Forward, Hybrid, Center) so the within-league position share is directly comparable across the two charts.

In [8]:
from plotly.subplots import make_subplots

POS_ORDER = ['Guard', 'Forward', 'Hybrid', 'Center']
POS_COLORS = {'Guard': '#66c2a5', 'Forward': '#fc8d62', 'Hybrid': '#8da0cb', 'Center': '#e78ac3'}

pos_counts = (
    combined.groupby(['League', 'PosGroup']).size()
    .unstack(fill_value=0)
    .reindex(columns=POS_ORDER, fill_value=0)
)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=['<b>NBA</b>', '<b>WNBA</b>'],
)
for i, league in enumerate(['NBA', 'WNBA'], start=1):
    fig.add_trace(
        go.Pie(
            labels=POS_ORDER,
            values=pos_counts.loc[league].tolist(),
            hole=0.45, sort=False,
            marker=dict(colors=[POS_COLORS[p] for p in POS_ORDER],
                        line=dict(color='white', width=2)),
            textinfo='label+percent',
            textfont=dict(size=13),
            hovertemplate='%{label}: %{value} players (%{percent})<extra></extra>',
        ),
        row=1, col=i,
    )

fig.update_layout(
    title=dict(
        text='<b>Position Composition by League</b><br>'
             '<sub style="color:#666">Each ring shows the position share within that league</sub>',
        x=0.02, xanchor='left',
    ),
    height=540, showlegend=False,
    margin=dict(t=110, l=30, r=30, b=30),
    font=BASE_FONT,
)
fig.show()

## Rebound analysis

A **rebound** is the act of recovering the ball after a missed shot. Every rebound is classified as one of two types:

- **ORB (Offensive Rebound)** — the rebounding team had just taken the shot, so they reclaim possession for a second-chance scoring opportunity.
- **DRB (Defensive Rebound)** — the rebounding team had been defending, so collecting the rebound ends the opponent's possession and lets the team push into offence.
- **TRB (Total Rebounds)** = ORB + DRB, the total number of rebounds a player records per game.

Rebounding is a strong indicator of physical effort, positioning, and the value each league places on second-chance possessions. The next five plots compare these three rebound metrics between the NBA and WNBA across players, positions, and rebound type.

### 6. Total rebounds per game — every player as a point

Horizontal strip plot with one dot per player and a dashed vertical line marking each league's mean — useful for seeing both the typical rebounding rate and the spread of outliers without smoothing the data the way a violin or KDE would.

In [9]:
fig = px.strip(
    combined, x='TRB', y='League', color='League',
    color_discrete_map=LEAGUE_COLORS,
    hover_data=['Player', 'Pos', 'ORB', 'DRB'],
    stripmode='overlay',
)
fig.update_traces(jitter=0.4, marker=dict(size=7, opacity=0.55, line=dict(width=0.5, color='white')))

means_trb = combined.groupby('League')['TRB'].mean().round(2)
for league, m in means_trb.items():
    fig.add_vline(
        x=m, line_dash='dot', line_color=LEAGUE_COLORS[league],
        line_width=2, opacity=0.85,
        annotation_text=f'{league} mean: {m}',
        annotation_position='top',
    )

fig.update_layout(
    title=dict(
        text='<b>Total Rebounds per Game — One Dot per Player</b><br>'
             '<sub style="color:#666">Dotted lines mark the per-league mean</sub>',
        x=0.02, xanchor='left',
    ),
    xaxis_title='TRB per game', yaxis_title='',
    height=440, showlegend=False,
    margin=dict(t=100, l=70, r=30, b=40),
    font=BASE_FONT,
)
fig.show()

### 7. Offensive vs defensive rebounds — bubble chart

Bubble scatter of ORB against DRB per game with marker size encoding total rebounds, so dominant rebounders pop visually and you can spot whether elite WNBA rebounders occupy the same corner of the plane as elite NBA rebounders.

In [10]:
fig = px.scatter(
    combined, x='ORB', y='DRB', color='League',
    size='TRB', size_max=24,
    color_discrete_map=LEAGUE_COLORS,
    hover_data=['Player', 'Pos', 'TRB'],
    opacity=0.6,
)
fig.update_traces(marker=dict(line=dict(width=0.5, color='white')))
fig.update_layout(
    title=dict(
        text='<b>Offensive vs Defensive Rebounds</b><br>'
             '<sub style="color:#666">Bubble size = total rebounds per game</sub>',
        x=0.02, xanchor='left',
    ),
    xaxis_title='ORB per game', yaxis_title='DRB per game',
    height=620,
    margin=dict(t=100, l=70, r=30, b=40),
    font=BASE_FONT,
)
fig.show()

### 8. Mean rebound rate — NBA vs WNBA dumbbell

Dumbbell chart pairing the league means for ORB and DRB and connecting them with a line, where the distance between the two markers is a direct visual of how big the gap is between the leagues on each rebound type.

In [11]:
means_reb = combined.groupby('League')[['ORB', 'DRB']].mean().round(2)
cats = ['ORB', 'DRB']
nba_vals = [means_reb.loc['NBA', c] for c in cats]
wnba_vals = [means_reb.loc['WNBA', c] for c in cats]

fig = go.Figure()
for i, cat in enumerate(cats):
    fig.add_trace(go.Scatter(
        x=[nba_vals[i], wnba_vals[i]], y=[cat, cat],
        mode='lines', line=dict(color='#cccccc', width=5),
        showlegend=False, hoverinfo='skip',
    ))
fig.add_trace(go.Scatter(
    x=nba_vals, y=cats, mode='markers+text', name='NBA',
    marker=dict(size=26, color=LEAGUE_COLORS['NBA'], line=dict(color='white', width=2)),
    text=[f'<b>{v}</b>' for v in nba_vals], textposition='top center',
    textfont=dict(size=12, color='#222'),
))
fig.add_trace(go.Scatter(
    x=wnba_vals, y=cats, mode='markers+text', name='WNBA',
    marker=dict(size=26, color=LEAGUE_COLORS['WNBA'], line=dict(color='white', width=2)),
    text=[f'<b>{v}</b>' for v in wnba_vals], textposition='bottom center',
    textfont=dict(size=12, color='#222'),
))
fig.update_layout(
    title=dict(
        text='<b>Mean Rebound Rate — NBA vs WNBA</b><br>'
             '<sub style="color:#666">Length of the grey connector = gap between the two leagues</sub>',
        x=0.02, xanchor='left',
    ),
    xaxis_title='Rebounds per game', yaxis_title='',
    height=460,
    margin=dict(t=100, l=80, r=40, b=50),
    font=BASE_FONT,
)
fig.show()

### 9. Offensive rebound share

Box plot of each player's offensive rebound share (ORB / TRB) by league, which is a style indicator: a higher share suggests a league more willing to crash the offensive glass for second-chance points rather than retreat into transition defence.

In [12]:
share_df = combined.dropna(subset=['ORB_share'])
fig = px.box(
    share_df, x='League', y='ORB_share', color='League',
    color_discrete_map=LEAGUE_COLORS,
    points='all', hover_data=['Player', 'Pos', 'ORB', 'DRB', 'TRB'],
)
fig.update_traces(marker=dict(size=4, opacity=0.55), jitter=0.4, pointpos=0)

share_means = share_df.groupby('League')['ORB_share'].mean()
for league, m in share_means.items():
    fig.add_annotation(
        x=league, y=m, text=f'<b>μ = {m*100:.0f}%</b>',
        showarrow=False, yshift=18,
        font=dict(size=13, color='#222'),
        bgcolor='rgba(255,255,255,0.92)', bordercolor='#666', borderwidth=1, borderpad=4,
    )

fig.update_layout(
    title=dict(
        text='<b>Offensive Rebound Share (ORB / TRB) by League</b><br>'
             '<sub style="color:#666">Higher share = more willingness to crash the offensive glass</sub>',
        x=0.02, xanchor='left',
    ),
    yaxis_title='ORB / TRB', xaxis_title='',
    height=540, showlegend=False, yaxis_tickformat='.0%',
    margin=dict(t=100, l=70, r=30, b=40),
    font=BASE_FONT,
)
fig.show()

### 10. Average rebounds by position and league — heatmap

Heatmap of the mean total rebounds per game broken down by position group and league, with cell colour intensity (and overlaid numbers) showing exactly which roles drive the rebound difference between the NBA and WNBA.

In [13]:
heat = (
    combined.groupby(['PosGroup', 'League'])['TRB'].mean().round(2)
    .reset_index()
    .pivot(index='PosGroup', columns='League', values='TRB')
    .reindex(['Guard', 'Forward', 'Hybrid', 'Center'])
)
fig = px.imshow(
    heat, text_auto=True,
    color_continuous_scale='OrRd', aspect='auto',
    labels=dict(x='League', y='Position group', color='TRB / game'),
)
fig.update_traces(textfont=dict(size=15, color='#222'))
fig.update_layout(
    title=dict(
        text='<b>Average Total Rebounds per Game by Position and League</b><br>'
             '<sub style="color:#666">Darker shade = more rebounds per game</sub>',
        x=0.02, xanchor='left',
    ),
    height=500,
    margin=dict(t=100, l=90, r=40, b=40),
    font=BASE_FONT,
)
fig.show()

## Defensive activity — blocks and steals

Steals (STL) and blocks (BLK) are the two defensive counting stats most directly tied to playing style: steals reflect perimeter aggression and anticipation, while blocks reflect rim protection and verticality. Plotting them jointly reveals where each league's defenders cluster — and any wing-vs-big specialisation pattern.

### 11. Steals vs blocks — bivariate KDE contours (with league toggle)

Seaborn-style bivariate kernel density estimate: smooth contour bands shade the regions of the STL × BLK plane where players are most concentrated, with a low density threshold trimming away the long thin tails so the chart focuses on the bulk of each league; use the dropdown in the top-right to switch between the NBA and the WNBA.

In [14]:
def league_scale(hex_color):
    r = int(hex_color[1:3], 16)
    g = int(hex_color[3:5], 16)
    b = int(hex_color[5:7], 16)
    return [
        [0.0,  f'rgba({r},{g},{b},0)'],
        [0.3,  f'rgba({r},{g},{b},0.35)'],
        [0.7,  f'rgba({r},{g},{b},0.75)'],
        [1.0,  f'rgba({r},{g},{b},1)'],
    ]

x_max = 1.8
y_max = 1.4

xgrid = np.linspace(0, x_max, 90)
ygrid = np.linspace(0, y_max, 90)
X, Y = np.meshgrid(xgrid, ygrid)
grid_pts = np.vstack([X.ravel(), Y.ravel()])

fig = go.Figure()
for league in ['NBA', 'WNBA']:
    sub = combined[combined['League'] == league].dropna(subset=['STL', 'BLK'])
    is_default = (league == 'NBA')
    color = LEAGUE_COLORS[league]

    kde = gaussian_kde(np.vstack([sub['STL'].values, sub['BLK'].values]))
    Z = kde(grid_pts).reshape(X.shape)
    thresh = Z.max() * 0.10

    fig.add_trace(go.Contour(
        x=xgrid, y=ygrid, z=Z,
        colorscale=league_scale(color),
        showscale=False,
        contours=dict(
            coloring='fill', showlines=True,
            start=thresh, end=Z.max(), size=(Z.max() - thresh) / 7,
        ),
        line=dict(width=0.6, color='white'),
        visible=is_default,
        hovertemplate='STL: %{x:.2f}<br>BLK: %{y:.2f}<br>Density: %{z:.3f}<extra></extra>',
        name=f'{league} KDE',
    ))

visibility = {'NBA': [True, False], 'WNBA': [False, True]}
subtitle = (f'<sub style="color:#666">Smoothed bivariate KDE (density ≥ 10% of peak) · '
            f'axes fixed at STL ∈ [0, {x_max}], BLK ∈ [0, {y_max}]</sub>')

fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=[
            dict(label='NBA', method='update',
                 args=[{'visible': visibility['NBA']},
                       {'title.text': f'<b>Steals vs Blocks per Game — NBA</b><br>{subtitle}'}]),
            dict(label='WNBA', method='update',
                 args=[{'visible': visibility['WNBA']},
                       {'title.text': f'<b>Steals vs Blocks per Game — WNBA</b><br>{subtitle}'}]),
        ],
        direction='down', showactive=True,
        x=1.0, y=1.15, xanchor='right', yanchor='top',
        bgcolor='#f7f7f7', bordercolor='#888', borderwidth=1, pad=dict(t=4, b=4, l=8, r=8),
    )],
    annotations=[dict(
        text='League:', x=0.86, y=1.145, xref='paper', yref='paper',
        xanchor='right', yanchor='top', showarrow=False,
        font=dict(size=13, color='#222'),
    )],
    title=dict(
        text=f'<b>Steals vs Blocks per Game — NBA</b><br>{subtitle}',
        x=0.02, xanchor='left',
    ),
    xaxis=dict(title='STL per game', range=[0, x_max], autorange=False, rangemode='nonnegative'),
    yaxis=dict(title='BLK per game', range=[0, y_max], autorange=False, rangemode='nonnegative'),
    height=600, showlegend=False,
    margin=dict(t=130, l=80, r=40, b=60),
    font=BASE_FONT,
)
fig.show()

### 12. Defensive output vs playing time — scatter with trendline (with panel dropdown)

Single scatter plot of minutes played per game against a defensive stat, with an OLS trendline overlaid and the regression equation and Pearson correlation `r` printed in the subtitle; use the dropdown in the top-right to switch between the four league × stat combinations (NBA Steals, NBA Blocks, WNBA Steals, WNBA Blocks) one at a time.

In [15]:
panel_lookup = [
    ('NBA',  'STL'),
    ('NBA',  'BLK'),
    ('WNBA', 'STL'),
    ('WNBA', 'BLK'),
]

results = {}
for league, stat in panel_lookup:
    sub = combined[combined['League'] == league].dropna(subset=['MP', stat])
    x_vals = sub['MP'].values
    y_vals = sub[stat].values
    slope, intercept = np.polyfit(x_vals, y_vals, 1)
    r = np.corrcoef(x_vals, y_vals)[0, 1]
    results[(league, stat)] = (slope, intercept, r, sub)

x_max = combined['MP'].max() * 1.05
y_max = {
    'STL': combined['STL'].quantile(0.99) + 0.2,
    'BLK': combined['BLK'].quantile(0.99) + 0.2,
}

fig = go.Figure()

for league, stat in panel_lookup:
    slope, intercept, r, sub = results[(league, stat)]
    color = LEAGUE_COLORS[league]
    is_default = (league == 'NBA' and stat == 'STL')

    fig.add_trace(go.Scatter(
        x=sub['MP'], y=sub[stat],
        mode='markers',
        marker=dict(size=7, color=color, opacity=0.6,
                    line=dict(width=0.5, color='white')),
        customdata=sub[['Player', 'Pos']].values,
        hovertemplate=('<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
                       'Minutes: %{x:.1f}<br>' + stat + ': %{y:.2f}<extra></extra>'),
        visible=is_default,
        showlegend=False,
        name=f'{league} {stat} dots',
    ))

    xline = np.linspace(sub['MP'].min(), sub['MP'].max(), 50)
    yline = slope * xline + intercept
    fig.add_trace(go.Scatter(
        x=xline, y=yline, mode='lines',
        line=dict(color='#222', width=2.5),
        visible=is_default,
        showlegend=False,
        hovertemplate=(f'slope = {slope:.4f}<br>'
                       f'intercept = {intercept:.3f}<br>'
                       f'r = {r:.2f}<extra></extra>'),
        name=f'{league} {stat} fit',
    ))

def build_title(league, stat):
    slope, intercept, r, _ = results[(league, stat)]
    full = {'STL': 'Steals', 'BLK': 'Blocks'}[stat]
    return (f'<b>{league} — {full} per Game vs Minutes Played</b><br>'
            f'<sub style="color:#666">'
            f'y = {slope:.3f}·x + {intercept:.2f}  ·  r = {r:.2f}</sub>')

buttons = []
for idx, (league, stat) in enumerate(panel_lookup):
    visibility = [False] * 8
    visibility[idx * 2] = True
    visibility[idx * 2 + 1] = True
    full = {'STL': 'Steals', 'BLK': 'Blocks'}[stat]
    buttons.append(dict(
        label=f'{league} — {full}',
        method='update',
        args=[
            {'visible': visibility},
            {
                'title.text': build_title(league, stat),
                'yaxis.title.text': f'{stat} per game',
                'yaxis.range': [0, y_max[stat]],
            },
        ],
    ))

fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction='down', showactive=True,
        x=1.0, y=1.15, xanchor='right', yanchor='top',
        bgcolor='#f7f7f7', bordercolor='#888', borderwidth=1, pad=dict(t=4, b=4, l=8, r=8),
    )],
    annotations=[dict(
        text='Panel:', x=0.78, y=1.145, xref='paper', yref='paper',
        xanchor='right', yanchor='top', showarrow=False,
        font=dict(size=13, color='#222'),
    )],
    title=dict(text=build_title('NBA', 'STL'), x=0.02, xanchor='left'),
    xaxis=dict(title='Minutes played per game', range=[0, x_max]),
    yaxis=dict(title='STL per game', range=[0, y_max['STL']]),
    height=620,
    margin=dict(t=130, l=80, r=40, b=60),
    font=BASE_FONT,
)
fig.show()